# PCA 如何降维与白化？

**面试回答主线：**PCA 在中心化数据上选择方差最大的正交方向，前 k 个方向提供最小平方重建误差的线性低维表示；白化再将每个主成分除以标准差，使协方差接近单位阵。高方差不等于高业务价值，白化也可能放大小特征值方向的噪声。本实验手写协方差、特征分解、重建与稳定白化。

## 真实案例

设备健康监测记录温度、功耗和电流。正常设备的三个数值高度相关，故障设备在“电流偏离温度/功耗关系”上异常。目标不是用原始绝对大小打分，而是识别偏离正常相关结构的设备。

In [1]:
import numpy as np  # 导入 NumPy 以手写 PCA 和白化。
np.set_printoptions(precision=3, suppress=True)  # 设置紧凑的矩阵显示。
device = np.array(['M01', 'M02', 'M03', 'M04', 'M05', 'M06', 'M07', 'M08', 'N01', 'N02'])  # 构造具名设备编号。
sensor = np.array([[20, 100, 10.0], [22, 110, 11.0], [24, 121, 12.1], [26, 130, 13.0], [28, 141, 14.2], [30, 151, 15.1], [32, 161, 16.2], [34, 171, 17.1], [27, 135, 20.5], [33, 165, 9.0]], dtype=float)  # 记录温度、功耗和电流，其中后两台是结构异常设备。
normal_index = np.arange(8)  # 将前八台已确认正常设备作为 PCA 拟合库。
all_index = np.arange(10)  # 保留全部设备用于异常评分。
print('设备 | 温度℃ | 功耗W | 电流A | 角色')  # 输出传感器账本表头。
for index in all_index:  # 逐条展示设备观测。
    role = '正常拟合' if index in normal_index else '待检测'  # 标识设备是否参与正常分布拟合。
    print(f'{device[index]} | {sensor[index, 0]:6.1f} | {sensor[index, 1]:5.1f} | {sensor[index, 2]:5.1f} | {role}')  # 输出一条设备记录。

设备 | 温度℃ | 功耗W | 电流A | 角色
M01 |   20.0 | 100.0 |  10.0 | 正常拟合
M02 |   22.0 | 110.0 |  11.0 | 正常拟合
M03 |   24.0 | 121.0 |  12.1 | 正常拟合
M04 |   26.0 | 130.0 |  13.0 | 正常拟合
M05 |   28.0 | 141.0 |  14.2 | 正常拟合
M06 |   30.0 | 151.0 |  15.1 | 正常拟合
M07 |   32.0 | 161.0 |  16.2 | 正常拟合
M08 |   34.0 | 171.0 |  17.1 | 正常拟合
N01 |   27.0 | 135.0 |  20.5 | 待检测
N02 |   33.0 | 165.0 |   9.0 | 待检测


## Baseline / 基线

基线以温度最高的设备为异常。它忽略传感器之间的相关结构，因此会把高负载但关系正常的设备误判，也会漏掉温度普通但电流异常的设备。

In [2]:
temperature_threshold = sensor[normal_index, 0].mean() + 1.5 * sensor[normal_index, 0].std()  # 根据正常设备温度构造简单阈值。
baseline_flag = sensor[:, 0] > temperature_threshold  # 仅根据温度判断异常。
print(f'温度异常阈值={temperature_threshold:.2f}℃')  # 输出基线阈值。
print('温度基线标记:', device[baseline_flag].tolist())  # 输出被单变量规则标记的设备。
print('问题：N01 的温度并不高，却有明显的电流结构偏离。')  # 解释基线漏检的业务语义。

温度异常阈值=33.87℃
温度基线标记: ['M08']
问题：N01 的温度并不高，却有明显的电流结构偏离。


In [3]:
normal_sensor = sensor[normal_index]  # 取出只用于拟合正常结构的传感器数据。
mean = normal_sensor.mean(axis=0)  # 计算正常设备的训练均值。
centered_normal = normal_sensor - mean  # 按训练均值中心化正常设备。
covariance = centered_normal.T @ centered_normal / (len(normal_sensor) - 1)  # 手写样本协方差矩阵。
eigenvalue, eigenvector = np.linalg.eigh(covariance)  # 对对称协方差矩阵做特征分解。
order = np.argsort(eigenvalue)[::-1]  # 将主成分按解释方差从大到小排序。
eigenvalue = eigenvalue[order]  # 重排特征值。
eigenvector = eigenvector[:, order]  # 同步重排主轴方向。
explained_ratio = eigenvalue / eigenvalue.sum()  # 计算每个主成分的解释方差比例。
print('协方差矩阵=', np.round(covariance, 3))  # 输出 PCA 的核心输入矩阵。
print('特征值:', np.round(eigenvalue, 4))  # 输出各主成分方差。
print('解释方差比:', np.round(explained_ratio, 4))  # 输出选择保留维数的中间依据。

协方差矩阵= [[ 24.    121.857  12.271]
 [121.857 618.839  62.323]
 [ 12.271  62.323   6.278]]
特征值: [649.111   0.005   0.002]
解释方差比: [1. 0. 0.]


In [4]:
centered_all = sensor - mean  # 使用正常训练均值中心化所有待评分设备。
component = centered_all @ eigenvector  # 将设备投影到 PCA 主成分坐标。
principal_vector = eigenvector[:, :1]  # 只保留解释方差最大的第一主成分。
reconstruction = (centered_all @ principal_vector) @ principal_vector.T  # 用一维 PCA 子空间重建中心化数据。
residual = centered_all - reconstruction  # 计算被丢弃子空间中的重建残差。
reconstruction_error = (residual ** 2).sum(axis=1)  # 将残差平方和作为结构异常分数。
threshold = np.quantile(reconstruction_error[normal_index], 0.95)  # 仅利用正常训练设备定义异常阈值。
pca_flag = reconstruction_error > threshold  # 标记偏离正常相关结构的设备。
print('第一主成分载荷 [温度, 功耗, 电流]:', np.round(principal_vector.reshape(-1), 3))  # 输出可解释的主轴方向。
print('重建误差:', np.round(reconstruction_error, 3))  # 输出每台设备的异常中间分数。
print('PCA 标记:', device[pca_flag].tolist())  # 输出基于结构残差的待检测设备。

第一主成分载荷 [温度, 功耗, 电流]: [-0.192 -0.976 -0.098]
重建误差: [ 0.     0.002  0.014  0.012  0.008  0.002  0.003  0.004 48.168 56.496]
PCA 标记: ['M03', 'N01', 'N02']


In [5]:
epsilon = 0.05  # 设置白化稳定项以避免小特征值放大噪声。
white_component = component / np.sqrt(eigenvalue + epsilon)  # 将每个 PCA 坐标除以稳定后的标准差。
white_normal = white_component[normal_index]  # 取出白化后的正常设备表示。
white_covariance = white_normal.T @ white_normal / (len(white_normal) - 1)  # 计算白化后正常数据的协方差。
print('白化后正常协方差=', np.round(white_covariance, 3))  # 输出接近单位阵的白化结果。
print('说明：epsilon 使协方差不必精确为单位阵，却换来小方差方向的数值稳定。')  # 解释稳定白化的工程取舍。

白化后正常协方差= [[ 1.    -0.     0.   ]
 [-0.     0.089  0.   ]
 [ 0.     0.     0.031]]
说明：epsilon 使协方差不必精确为单位阵，却换来小方差方向的数值稳定。


## 结果解读

第一主成分压缩了温度、功耗和电流共同上升的正常负载趋势；重建误差则关注未被该趋势解释的部分，因此可发现 N01/N02 的结构偏离。累计解释方差只能说明压缩程度，不能单独证明故障检测效果。

In [6]:
print('设备 | 温度基线 | PCA重建误差 | PCA异常')  # 输出两种异常策略的逐设备对照表头。
for index in all_index:  # 逐条展示单变量和结构异常结论。
    print(f'{device[index]} | {str(bool(baseline_flag[index])):8s} | {reconstruction_error[index]:11.3f} | {bool(pca_flag[index])}')  # 输出一条设备的结果。
print('教学结论：PCA 是线性相关结构模型；异常仍需由有标签回放、人工复核或下游损失验证。')  # 明确 PCA 不会自动给出业务真相。

设备 | 温度基线 | PCA重建误差 | PCA异常
M01 | False    |       0.000 | False
M02 | False    |       0.002 | False
M03 | False    |       0.014 | True
M04 | False    |       0.012 | False
M05 | False    |       0.008 | False
M06 | False    |       0.002 | False
M07 | False    |       0.003 | False
M08 | True     |       0.004 | False
N01 | False    |      48.168 | True
N02 | False    |      56.496 | True
教学结论：PCA 是线性相关结构模型；异常仍需由有标签回放、人工复核或下游损失验证。


## 失败案例与修复

失败做法一：用全体设备（包含待检测异常）拟合主轴，异常会把“正常方向”拉偏。失败做法二：白化时直接除以极小特征值，会夸大传感器噪声。修复是只用确认正常的训练窗口拟合，并加入 epsilon 或截断小主成分。

In [7]:
leaky_mean = sensor.mean(axis=0)  # 错误地让待检测异常设备参与 PCA 均值拟合。
leaky_centered = sensor - leaky_mean  # 使用受异常污染的均值中心化全部设备。
leaky_covariance = leaky_centered.T @ leaky_centered / (len(sensor) - 1)  # 计算受污染的协方差矩阵。
leaky_value, leaky_vector = np.linalg.eigh(leaky_covariance)  # 分解受污染协方差。
leaky_axis = leaky_vector[:, np.argmax(leaky_value)]  # 取被异常拉偏后的第一主轴。
bad_white_scale = 1.0 / np.sqrt(eigenvalue[-1])  # 故意展示不加 epsilon 的小方差放大倍率。
safe_white_scale = 1.0 / np.sqrt(eigenvalue[-1] + epsilon)  # 计算加入稳定项后的放大倍率。
print('失败：全体拟合第一主轴:', np.round(leaky_axis, 3))  # 输出异常污染后的方向。
print('修复：正常库拟合第一主轴:', np.round(principal_vector.reshape(-1), 3))  # 输出只用正常设备的方向。
print(f'最小主成分放大倍数：无epsilon={bad_white_scale:.2f}，有epsilon={safe_white_scale:.2f}')  # 展示白化稳定性改进。
print('生产差距：需维护正常样本定义、在线漂移检测、传感器校准、阈值版本和人工工单闭环。')  # 描述真实监控系统的补充机制。

失败：全体拟合第一主轴: [-0.194 -0.979 -0.057]
修复：正常库拟合第一主轴: [-0.192 -0.976 -0.098]
最小主成分放大倍数：无epsilon=24.98，有epsilon=4.40
生产差距：需维护正常样本定义、在线漂移检测、传感器校准、阈值版本和人工工单闭环。


In [8]:
assert len(device) >= 5  # 保护案例包含至少五台设备。
assert explained_ratio[0] > 0.9  # 保护正常设备的主要变化确实近似低维。
assert 'N01' in device[pca_flag].tolist()  # 保护 PCA 残差发现温度基线漏掉的结构异常。
assert safe_white_scale < bad_white_scale  # 保护 epsilon 会降低小特征值方向的噪声放大。